In [ ]:
import numpy as np

class LogisticRegression:
    def __init__(self, lr=0.1, epochs=1000, tol=1e-6):
        self.lr = lr
        self.epochs = epochs
        self.tol = tol
        self.weights = None
        self.bias = 0.0
        self.losses = []

    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def _compute_loss(self, y, ypred):
        eps = 1e-15
        ypred_c = np.clip(ypred, eps, 1.0 - eps)
        return -np.mean(y * np.log(ypred_c) + (1.0 - y) * np.log(1.0 - ypred_c))

    def fit(self, X, y, solver='gd'):
        """
        solver: 'gd' (Gradient Descent) or 'newton' (Newton-Raphson / IRLS)
        """
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        samples, features = X.shape
        self.losses = []

        if solver == 'gd':
            self.weights = np.zeros(features)
            self.bias = 0.0

            for _ in range(self.epochs):
                z = X @ self.weights + self.bias
                ypred = self.sigmoid(z)
                self.losses.append(self._compute_loss(y, ypred))

                # Gradients
                error = ypred - y
                dw = (1 / samples) * (X.T @ error)
                db = (1 / samples) * np.sum(error)

                self.weights -= self.lr * dw
                self.bias -= self.lr * db

        elif solver == 'newton':
            # Augment X with a column of 1s to combine weights and bias into theta: shape (N, D + 1)
            X_b = np.c_[X, np.ones(samples)]
            theta = np.zeros(features + 1)

            for i in range(self.epochs):
                z = X_b @ theta
                ypred = self.sigmoid(z)
                self.losses.append(self._compute_loss(y, ypred))

                # Gradient: g = (1 / N) * X_b^T (ypred - y)
                g = (1 / samples) * (X_b.T @ (ypred - y))

                # Variance weighting vector: s_i = ypred_i * (1 - ypred_i)
                s = ypred * (1.0 - ypred)
                
                # Hessian: H = (1 / N) * X_b^T @ diag(s) @ X_b
                # Vectorized without building full N x N diagonal matrix:
                H = (1 / samples) * (X_b.T @ (s[:, np.newaxis] * X_b))
                
                # Add tiny Tikhonov regularization (ridge) for numerical invertibility
                H += 1e-6 * np.eye(features + 1)

                # Solve H * delta_theta = g  (equivalent to delta_theta = inv(H) @ g)
                delta_theta = np.linalg.solve(H, g)

                # Newton update
                theta -= delta_theta

                # Check convergence
                if np.linalg.norm(delta_theta) < self.tol:
                    break

            self.weights = theta[:-1]
            self.bias = theta[-1]

        else:
            raise ValueError("solver must be 'gd' or 'newton'")

        return self

    def predictprob(self, X):
        X = np.asarray(X, dtype=float)
        return self.sigmoid(X @ self.weights + self.bias)

    def predict(self, X, t=0.5):
        return (self.predictprob(X) >= t).astype(int)